### Transform Orders Data - Explode Arrays
1. Access elements from the JSON object
2. Deduplicate Array Elements
3. Explode Arrays
4. Write the Transformed Data to Silver

In [0]:
%sql
SELECT * FROM gizmobox.silver.orders_json

### Access elements from the JSON object

In [0]:
%sql
SELECT
json_value.customer_id,
json_value.order_date,
json_value.order_id,
json_value.order_status,
json_value.payment_method,
json_value.total_amount,
json_value.items,
json_value.transaction_timestamp
FROM gizmobox.silver.orders_json

### Deduplicate Array Elements

In [0]:
%sql
SELECT
json_value.customer_id,
json_value.order_date,
json_value.order_id,
json_value.order_status,
json_value.payment_method,
json_value.total_amount,
array_distinct(json_value.items) as items,
json_value.transaction_timestamp
FROM gizmobox.silver.orders_json

### Explode Arrays

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW tv_orders_explode AS
SELECT
json_value.customer_id,
json_value.order_date,
json_value.order_id,
json_value.order_status,
json_value.payment_method,
json_value.total_amount,
explode(array_distinct(json_value.items)) as items,
json_value.transaction_timestamp
FROM gizmobox.silver.orders_json

In [0]:
%sql
SELECT * FROM tv_orders_explode

### Write the Transformed Data to Silver

In [0]:
%sql
CREATE TABLE gizmobox.silver.orders AS
SELECT 
  customer_id,
  order_date,
  order_id,
  order_status,
  payment_method,
  total_amount,
  items.item_id,
  items.name,
  items.category,
  items.details.brand,
  items.details.color,
  items.quantity,
  items.price,
  transaction_timestamp
FROM tv_orders_explode

In [0]:
%sql
SELECT * FROM gizmobox.silver.orders